# **BSM Event Generation Tutorial**

**Written by**: 
- Tony Menzo (U. of Alabama / Fermilab)

This notebook demonstrates interactive, iterated conversations with an LLM agent to perform a complete Beyond Standard Model (BSM) physics workflow:

1. **FeynRules**: Define BSM model and convert to UFO format
2. **MadGraph**: Generate parton-level events
3. **Pythia**: Shower and hadronize events
4. **Analysis**: Reconstruct objects and compute observables

**What you'll learn:**
1. Setting up an agent with BSM physics tools
2. Interactive multi-turn conversations with the agent
3. Complete workflow: FeynRules $\to$ MadGraph $\to$ Pythia $\to$ Jets $\to$ Analysis

**Physics scenario:** Scalar leptoquark $S_1 \sim (\bar{\mathbf{3}}, \mathbf{1}, 1/3)$ pair production at the LHC
- Lagrangian:
    $$
    \mathcal{L} \supset (D_\mu S_1)^\dagger (D^\mu S_1) + y_{ij}\,\overline{u^c_{Ri}}\, e_{Rj}\, S_1 + \mathrm{h.c.}
    $$
- Process: $pp \to S_1 \bar{S}_1 \to (e^- u)(e^+ \bar{u})$
- Signature: Opposite-sign dileptons + dijets

---

This tutorial is built using **HEPTAPOD**, a toolkit for agentic high-energy physics workflows, built on the [Orchestral](https://orchestral-ai.com) orchestration engine.

**Repository**: [https://github.com/tonymenzo/heptapod](https://github.com/tonymenzo/heptapod)

## 0. Prerequisites

**Required:** HEPTAPOD repository cloned and dependencies installed.

**This tutorial requires:**
- FeynRules + Mathematica/WolframScript
- MadGraph5_aMC@NLO  
- Pythia 8

For detailed setup instructions, see [heptapod_setup.ipynb](../../setup/heptapod_setup.ipynb).

### **Environment Setup**

In [ ]:
# ============================================================================
# SETUP - Run this cell first
# ============================================================================

import sys
from pathlib import Path

# Add repository root to path (examples/workflows/bsm/ -> 3 levels up)
REPO_ROOT = Path.cwd().parent.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Now import and run shared setup
from examples.shared.heptapod_setup import setup_heptapod

# This displays configuration status and returns a dict
# Use verbose=False for silent setup
config = setup_heptapod(notebook_depth=3)

In [ ]:
# Core Orchestral imports
from orchestral import Agent
from orchestral.tools import (
    RunCommandTool, WriteFileTool, ReadFileTool, EditFileTool,
    FileSearchTool, FindFilesTool, RunPythonTool, WebSearchTool,
    TodoWrite, TodoRead
)
from orchestral.tools.hooks import TruncateOutputHook

# LLM provider imports
from orchestral.llm import GPT, Claude, Gemini, Groq
from llm import get_ollama

# HEPTAPOD physics tools
from tools.feynrules import FeynRulesToUFOTool
from tools.mg5 import MadGraphFromRunCardTool
from tools.pythia import PythiaFromRunCardTool, JetClusterSlowJetTool
from tools.analysis.conversions import EventJSONLToNumpyTool, JetsJSONLToNumpyTool
from tools.analysis.kinematics import (
    CalculateInvariantMassTool, CalculateTransverseMomentumTool,
    CalculateDeltaRTool, ApplyCutsTool, GetHardestNTool,
    GetHardestNJetsTool, FilterByPDGIDTool, SortByPtTool
)
from tools.analysis.reconstruction import ResonanceReconstructionTool

# Shared utilities
from examples.shared.tool_logger import ToolCallLogger

# Configuration
from config import feynrules_path, mg5_path, wolframscript_path

print("Imports successful!")
print(f"FeynRules path: {feynrules_path}")
print(f"MadGraph path: {mg5_path}")

## 1. Create Sandbox Workspace

We'll create a fresh sandbox directory containing template files:
- `feynrules/models/S1_LQ_RR.fr` - Scalar leptoquark model
- `madgraph/cards/S1_LQ_RR_pp_lqlq_scan.mg5` - MadGraph run card
- `pythia/cards/S1_LQ_RR_pp_ljlj.cmnd` - Pythia configuration

In [ ]:
# Import sandbox utilities
sys.path.insert(0, str(Path.cwd().parent.parent / 'shared'))
from sandbox_utils import create_new_sandbox

# Create a new sandbox with explorer mode (interactive)
demo_files_dir = Path.cwd() / 'sandbox'
base_directory, system_prompt = create_new_sandbox(demo_files_dir, mode="explorer")

print(f"\nSandbox created: {base_directory}")

In [ ]:
# Let's see what files are in our sandbox
import os

def show_tree(path, prefix=""):
    """Display directory tree."""
    entries = sorted(os.listdir(path))
    for i, entry in enumerate(entries):
        if entry.startswith('.'):
            continue
        full_path = os.path.join(path, entry)
        is_last = i == len(entries) - 1
        connector = "+-- " if is_last else "|-- "
        print(f"{prefix}{connector}{entry}")
        if os.path.isdir(full_path):
            extension = "    " if is_last else "|   "
            show_tree(full_path, prefix + extension)

print(f"Sandbox contents:")
print(base_directory.split('/')[-1] + "/")
show_tree(base_directory)

## 2. Configure the Agent

We'll set up an agent with all the tools needed for BSM event generation.

In [ ]:
# Define all available tools
tools = [
    # Core file/command tools
    RunCommandTool(base_directory=base_directory),
    WriteFileTool(base_directory=base_directory),
    ReadFileTool(base_directory=base_directory, show_line_numbers=True),
    EditFileTool(base_directory=base_directory),
    FindFilesTool(base_directory=base_directory),
    FileSearchTool(base_directory=base_directory),
    RunPythonTool(base_directory=base_directory, timeout=1000),
    WebSearchTool(),
    
    # Event generation pipeline
    FeynRulesToUFOTool(
        base_directory=base_directory,
        feynrules_path=feynrules_path,
        wolframscript_path=wolframscript_path
    ),
    MadGraphFromRunCardTool(base_directory=base_directory, mg5_path=mg5_path),
    PythiaFromRunCardTool(base_directory=base_directory),
    JetClusterSlowJetTool(base_directory=base_directory),
    
    # Data conversion
    EventJSONLToNumpyTool(base_directory=base_directory),
    JetsJSONLToNumpyTool(base_directory=base_directory),
    
    # Analysis - Kinematics
    CalculateInvariantMassTool(base_directory=base_directory),
    CalculateTransverseMomentumTool(base_directory=base_directory),
    CalculateDeltaRTool(base_directory=base_directory),
    ApplyCutsTool(base_directory=base_directory),
    
    # Analysis - Event selection
    GetHardestNTool(base_directory=base_directory),
    GetHardestNJetsTool(base_directory=base_directory),
    FilterByPDGIDTool(base_directory=base_directory),
    SortByPtTool(base_directory=base_directory),
    
    # Analysis - Reconstruction
    ResonanceReconstructionTool(base_directory=base_directory),
    
    # Task management
    TodoRead(),
    TodoWrite(base_directory=base_directory)
]

print(f"Configured {len(tools)} tools for BSM event generation")

### 2.1 Tool Call Logging (Optional)

The `ToolCallLogger` shows what tools the agent calls and their arguments. This is useful for debugging and understanding agent behavior.

In [ ]:
# Configure hooks
# Set ENABLE_TOOL_LOGGING = True to see all tool calls

ENABLE_TOOL_LOGGING = True  # <-- Toggle this to enable/disable tool call logging

hooks = [
    TruncateOutputHook(max_length=10000),  # Prevent huge outputs
]

if ENABLE_TOOL_LOGGING:
    # Add the logging hook - customize verbosity here
    hooks.append(ToolCallLogger(
        verbose=True,       # Show full arguments (False = just tool names)
        show_results=False  # Show tool outputs (can be very verbose)
    ))
    print("Tool call logging ENABLED - watch for >>> TOOL CALL messages")
else:
    print("Tool call logging DISABLED - set ENABLE_TOOL_LOGGING = True to enable")

In [ ]:
# Choose your LLM provider
# Uncomment ONE of the following:

# Option 1: OpenAI GPT (default)
llm = GPT()

# Option 2: Anthropic Claude
# llm = Claude()

# Option 3: Google Gemini
# llm = Gemini()

# Option 4: Groq (fast inference)
# llm = Groq()

# Option 5: Local Ollama
# llm = get_ollama()

print(f"Using LLM: {llm.__class__.__name__}")

In [ ]:
# Create the agent
agent = Agent(
    llm=llm,
    tools=tools,
    tool_hooks=hooks,
    system_prompt=system_prompt,
    debug=False  # Set True for even more detailed internal logs
)

print("Agent created successfully!")
print("\nThe agent is now ready to help with BSM event generation.")
print("It has access to FeynRules, MadGraph, Pythia, and analysis tools.")

## 3. Interactive Tutorial - Exploring the Workspace

Now we begin the interactive part. We'll have a multi-turn conversation with the agent, guiding it through the BSM workflow step by step.

**Key pattern:** Each `agent.run(prompt)` call is a turn in the conversation. The agent maintains context between calls.

### 3.1 Explore the Workspace

In [ ]:
# First interaction: Ask the agent to explore the workspace
response = agent.run(
    "Please explore the workspace and tell me what files are available. "
    "I'm interested in BSM physics event generation."
)
print(response)

### 3.2 Understand the Physics Model

In [ ]:
# Second interaction: Ask about the FeynRules model
response = agent.run(
    "Read the FeynRules model file and explain the physics. "
    "What particle is being added to the Standard Model? "
    "What are its quantum numbers and interactions?"
)
print(response)

## 4. Event Generation Pipeline

Now we'll guide the agent through the complete event generation chain.

### 4.1 Convert FeynRules Model to UFO Format

In [ ]:
# Generate UFO model from FeynRules
response = agent.run(
    "Convert the S1_LQ_RR.fr FeynRules model to UFO format. "
    "This will be needed for MadGraph event generation."
)
print(response)

### 4.2 Review the MadGraph Configuration

In [ ]:
# Review the MadGraph run card
response = agent.run(
    "Read the MadGraph run card in madgraph/cards/ and explain:\n"
    "1. What process is being generated?\n"
    "2. What are the beam energies?\n" 
    "3. What mass values are being scanned?\n"
    "4. How many events will be generated?"
)
print(response)

### 4.3 Generate Parton-Level Events with MadGraph

**Note:** This step may take several minutes depending on the number of events and mass points.

In [ ]:
# Generate events with MadGraph
# Note: Modify NEVENTS in the run card if you want fewer events for testing
response = agent.run(
    "Run MadGraph to generate parton-level events using the run card. "
    "Use the UFO model we just created. "
    "Start with just 100 events for a quick test (modify NEVENTS if needed)."
)
print(response)

### 4.4 Hadronization with Pythia

In [ ]:
# Run Pythia for showering and hadronization
response = agent.run(
    "Now run Pythia to shower and hadronize the MadGraph events. "
    "Use the Pythia card in pythia/cards/. "
    "Make sure to point it to the LHE file from MadGraph."
)
print(response)

### 4.5 Jet Clustering

After hadronization, we need to cluster the hadrons into jets. This reconstructs the quark-level objects from the spray of hadrons produced by Pythia.

In [ ]:
# Cluster hadrons into jets using the SlowJet algorithm
response = agent.run(
    "Cluster the hadronized Pythia events into jets using the jet clustering tool. "
    "Use anti-kT algorithm with R=0.4 (standard ATLAS/CMS choice). "
    "The output should contain reconstructed jets that we can use for analysis."
)
print(response)

## 5. Analysis

With events generated, we can now analyze them.

### 5.1 Convert Events to Analysis Format

In [ ]:
# Convert events to numpy format
response = agent.run(
    "Convert the Pythia output events to numpy format for analysis. "
    "This will make it easier to compute kinematic quantities."
)
print(response)

### 5.2 Reconstruct the Leptoquark Mass

In [ ]:
# Reconstruct the leptoquark from its decay products
response = agent.run(
    "The leptoquark decays as S1 -> e u. \n"
    "Reconstruct the leptoquark invariant mass by combining:\n"
    "- The hardest electron (PDG ID 11 or -11)\n"
    "- The hardest up-type jet (or just hardest jet)\n\n"
    "Calculate and report the reconstructed mass distribution."
)
print(response)

### 5.3 Apply Selection Cuts

In [ ]:
# Apply realistic selection cuts
response = agent.run(
    "Apply typical LHC selection cuts to the events:\n"
    "- Electron pT > 30 GeV\n"
    "- Electron |eta| < 2.5\n"
    "- Jet pT > 30 GeV\n"
    "- Jet |eta| < 4.0\n\n"
    "Report how many events pass the selection."
)
print(response)

### 5.4 Create Summary Plot

In [ ]:
# Ask the agent to create a summary plot
response = agent.run(
    "Create a histogram of the reconstructed leptoquark mass using matplotlib. "
    "Save it as 'leptoquark_mass.png' in the output directory. "
    "Include appropriate labels and title."
)
print(response)

In [ ]:
# Display the plot if it was created
from IPython.display import Image, display
import glob

plot_files = glob.glob(f"{base_directory}/**/leptoquark_mass.png", recursive=True)
if plot_files:
    display(Image(filename=plot_files[0]))
else:
    print("Plot not found - check the agent's response for the file location")

## 6. Custom Queries

The agent maintains conversation context. You can ask follow-up questions or request additional analyses.

In [ ]:
# Example: Request additional analysis
response = agent.run(
    "Calculate the transverse momentum distribution of the leading electron. "
    "Create a histogram and save it."
)
print(response)

In [ ]:
# Your own custom query - modify this!
custom_query = """
Summarize the complete workflow we've done:
1. What model did we use?
2. What process did we generate?
3. What were the key results?
"""

response = agent.run(custom_query)
print(response)

## 7. Web Interface (Optional)

For extended interactive sessions, you can use the web interface instead of notebook cells.

In [ ]:
import app.server as app_server

# UNCOMMENT to start web server:
# app_server.run_server(
#     agent,
#     host="127.0.0.1",
#     port=8000,
#     open_browser=True,
#     max_tool_iterations=100
# )

print("Uncomment above to start interactive web interface")

## Summary

This tutorial demonstrated a complete BSM event generation workflow:

### Workflow Steps
| Step | Tool | Description |
|------|------|-------------|
| 1 | FeynRules | Define BSM model (scalar leptoquark) -> UFO |
| 2 | MadGraph | Generate parton-level events |
| 3 | Pythia | Shower and hadronize events |
| 4 | SlowJet | Cluster hadrons into jets (anti-kT, R=0.4) |
| 5 | Analysis | Kinematics, cuts, mass reconstruction |

### Key Features
- **Multi-turn conversations**: Agent maintains context between calls
- **Tool logging**: See exactly what the agent does with `ToolCallLogger`
- **Complete pipeline**: From Lagrangian to histograms
- **Interactive guidance**: Control each step while agent handles details

### Physics Summary
- **Model**: Scalar leptoquark $S_1 \sim (\bar{\mathbf{3}}, \mathbf{1}, 1/3)$
- **Process**: $pp \to S_1 \bar{S}_1 \to (e^- u)(e^+ \bar{u})$
- **Signature**: Opposite-sign dileptons + dijets

**Next steps:**
- Modify the FeynRules model to study different BSM scenarios
- Change MadGraph parameters (mass, coupling, process)
- Add more sophisticated analysis cuts
- Compare with background processes

## Cleanup

In [ ]:
# Uncomment to remove sandbox
# import shutil
# if Path(base_directory).exists():
#     shutil.rmtree(base_directory)
#     print(f"Removed: {base_directory}")

print("Tutorial complete!")
print(f"\nGenerated files are in: {base_directory}")